# High-rise 05 - Facade Cp snapshots

Colour the body mesh by its per-triangle Cp statistics (mean / min / max
over the record) and render one image per facade. Facades are split by the
outward-normal direction of each triangle (`+x` / `-x` / `+y` / `-y` sides,
`+z` roof). Overview iso renders go to `deliverables/`; per-facade views go
to `debug/`.

Rendering uses a pure-matplotlib 3-D renderer so it runs headless. If the
optional `[vtk]` extra (PyVista) is installed, a contoured, colour-barred
snapshot is also written.

In [ ]:
import os
import pathlib

import numpy as np  # noqa: F401  (used across later cells)

from cfdmod import high_rise as hr
from cfdmod.high_rise import plotting


def _find_repo(start: pathlib.Path) -> pathlib.Path:
    p = start.resolve()
    while p != p.parent:
        if (p / "pyproject.toml").exists():
            return p
        p = p.parent
    return start.resolve()


REPO = _find_repo(pathlib.Path.cwd())

plotting.apply_style()

FIX = REPO / "fixtures" / "tests"
OUTPUT_BASE = pathlib.Path(
    os.environ.get("CFDMOD_HR_OUTPUT_BASE", REPO / "examples" / "high_rise" / "_run")
)
VERSION = os.environ.get("CFDMOD_HR_VERSION", "example")
print("REPO:", REPO)
print("OUTPUT_BASE:", OUTPUT_BASE, "| VERSION:", VERSION)

In [ ]:
from cfdmod.adapters.xdmf_h5 import XdmfH5Storage

# --- config -------------------------------------------------------------
MESH = pathlib.Path(
    os.environ.get("CFDMOD_HR_MESH", FIX / "pressure" / "galpao" / "galpao.normalized.lnas")
)
ARTIFACTS = OUTPUT_BASE / "artifacts" / VERSION
cp = XdmfH5Storage(ARTIFACTS).read_data_source(pathlib.Path("cp.time_series"))

geom = hr.snapshots.load_geometry(MESH)
n_tri = int(np.asarray(geom.triangle_vertices).shape[0])
cp_series = np.asarray(cp.fields.read("cp"))
cp_stats = {
    "mean": np.nanmean(cp_series, axis=1),
    "min": np.nanmin(cp_series, axis=1),
    "max": np.nanmax(cp_series, axis=1),
}
groups = hr.snapshots.facade_groups(MESH)
print("facades:", {k: len(v) for k, v in groups.items()})

In [ ]:
# --- render -------------------------------------------------------------
dbg = hr.DebugWriter(OUTPUT_BASE, stage="facade", version=VERSION)
clim = (float(np.nanmin(cp_stats["min"])), float(np.nanmax(cp_stats["max"])))

# Overview iso renders for each Cp statistic (engineer-facing).
for stat, vals in cp_stats.items():
    fig, _ = hr.snapshots.triangle_field_figure(
        geom, vals, view=hr.snapshots.STANDARD_VIEWS["iso"], clim=clim,
        title=f"{stat} Cp", cbar_label="Cp [-]",
    )
    dbg.savefig(fig, f"cp_{stat}_iso.png", deliverable=True)
    plotting.close(fig)

# Per-facade mean-Cp views (exploratory).
view_for = {"n_+x": "right", "n_-x": "left", "n_+y": "front", "n_-y": "back", "n_+z": "top"}
for name, idx in groups.items():
    fig, _ = hr.snapshots.triangle_field_figure(
        geom, cp_stats["mean"], subset=idx,
        view=hr.snapshots.STANDARD_VIEWS[view_for.get(name, "iso")], clim=clim,
        title=f"mean Cp - {hr.snapshots.FACADE_LABELS.get(name, name)}", cbar_label="Cp [-]",
    )
    dbg.savefig(fig, f"facade_{name}.png")
    plotting.close(fig)

# Optional high-quality PyVista render (only if the [vtk] extra is installed).
vtp = dbg.debug_path("cp_facades.vtp")
if hr.snapshots.write_field_vtp(geom, {"Cp_mean": cp_stats["mean"]}, vtp):
    ok = hr.snapshots.render_vtp_snapshot(
        vtp, dbg.deliverable_path("cp_mean_pyvista.png"),
        scalar="Cp_mean", label="mean Cp", clim=clim,
    )
    print("pyvista snapshot:", ok)
else:
    print("pyvista/[vtk] not installed - matplotlib facade images only")
print("facade images under", dbg.debug_dir)